# ECS171 Final Project "The Gold" 


In [ ]:
#install all dependencies
%pip install pandas numpy matplotlib seaborn scikit-learn scipy tensorflow

  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached scikit_learn-1.8.0-cp313-cp313-macosx_12_0_arm64.whl.metadata (11 kB)
  Using cached scipy-1.17.1-cp313-cp313-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached contourpy-1.3.3-cp313-cp313-macosx_11_0_arm64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached kiwisolver-1.5.0-cp313-cp313-macosx_11_0_arm64.whl.metadata (5.1 kB)
  Using cached pillow-12.2.0-cp313-cp313-macosx_11_0_arm64.whl.metadata (8.8 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached filelock-3.29.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9

In [ ]:
#CORE
import numpy as np
import pandas as pd
import itertools

#visualizations
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

#preprocessing
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

#Model / Neural Network
import tensorflow as tf

#Reproducability
import random
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(42)  # sets seeds for base-python, numpy and tf
tf.config.experimental.enable_op_determinism()


In [19]:
# data preprocessing

#load dataset from kaggle
df = pd.read_csv('Gold_vs_Economic_Factors_2015_2026.csv')
df = df.drop(columns=["Date"])  # removing date since each entry advances by one day
df_list = np.array(df.values.tolist(), dtype=float)  # converts csv object to a numpy array for easier handling

#inital look at dataset, types and missing values
print(f"Shape: {df.shape}")
print(f"Day range: {df.index.min()} to {df.index.max()}")
print(f"\n--- dtypes ---\n{df.dtypes}")
print(f"\n--- missing values ---\n{df.isnull().sum()}")

# Min Max Normalization
min_max_norm = MinMaxScaler()
min_max_norm_data = min_max_norm.fit_transform(df_list)
X = min_max_norm_data[:, np.newaxis, 0]  # converts gold price to a vertical list
min_max_norm_data = np.delete(min_max_norm_data, 1, axis=1)


Shape: (4137, 4)
Day range: 0 to 4136

--- dtypes ---
Gold_Price_XAU_USD     float64
US_Dollar_Index_DXY    float64
Crude_Oil_Price        float64
Inflation_Rate_Pct     float64
dtype: object

--- missing values ---
Gold_Price_XAU_USD     0
US_Dollar_Index_DXY    0
Crude_Oil_Price        0
Inflation_Rate_Pct     0
dtype: int64


### Data Visualization

In [ ]:
# data normalization
data = df.drop('Date', axis=1)
data_norm = (data - data.min()) / (data.max() - data.min())



# head data shown
print("\nNormalized Data:")
print(data_norm.head())

print("non Normalized Data")
print(data.head())

# data visualization
sns.pairplot(data_norm)
plt.show()


In [ ]:
# model

# Grid Search Hyperparameter Tuning
# possible hyperparameter values
hidden_1_layers = [2, 3, 4]
hidden_2_layers = [2, 3, 4]
learning_rates = [0.01, 0.03, 0.05]
momentums = [0.8, 0.9, 0.99]
epochs_list = [50, 100, 150]

best_mse = float("inf")
best_params = dict()

# try every combination of the above hyperparameters, grid search
for h1, h2, lr, momentum, epochs in itertools.product(
    hidden_1_layers, hidden_2_layers, learning_rates, momentums, epochs_list
):
    kfold = KFold(10)  # k = 10

    mse_list = []
    for train_index, test_index in kfold.split(min_max_norm_data):
        # get training and testing data based on k-fold
        x_train, x_test = min_max_norm_data[train_index], min_max_norm_data[test_index]
        y_train, y_test = X[train_index], X[test_index]
        
        # configuring neural network
        nn_model = tf.keras.Sequential([
            tf.keras.layers.Dense(3, activation=tf.keras.layers.LeakyReLU(alpha=0.1), input_shape=(3,)),
            tf.keras.layers.Dense(h1, activation=tf.keras.layers.LeakyReLU(alpha=0.1)),
            tf.keras.layers.Dense(h2, activation=tf.keras.layers.LeakyReLU(alpha=0.1)),
            tf.keras.layers.Dense(1, activation=tf.keras.layers.LeakyReLU(alpha=0.1))
        ])

        nn_model.compile(
            optimizer=tf.keras.optimizers.SGD(learning_rate=lr, momentum=momentum),  # stochastic gradient descent
            loss="mean_squared_error",
            metrics=["mse"]
        )

        model_hist = nn_model.fit(
            x_train,
            y_train,
            validation_data=(x_test, y_test),
            epochs=epochs,
            batch_size = 512
        )

        y_pred = nn_model.predict(x_test)
        curr_mse = mean_squared_error(y_test, y_pred)
        mse_list.append(curr_mse)

    avg_mse = np.mean(mse_list)

    # change optimal hyperparameters if MSE is the best so far
    if avg_mse < best_mse:
        best_mse = avg_mse

        best_params = {
            "hidden_1_layers": h1,
            "hidden_2_layers": h2,
            "learning_rate": lr,
            "momentum": momentum,
            "epoch_num": epochs
        }

In [ ]:
# results

print(best_params)

[[0.03227456 0.         0.91714286]
 [0.03198468 0.00206751 0.66714286]
 [0.03425813 0.01413597 0.47142857]
 ...
 [0.93357154 0.96975671 0.13857143]
 [0.93246738 0.96102991 0.36571429]
 [0.93244784 0.96891528 0.49285714]]
NaNs in y_true: True
